# NIH ChestX-ray14 — HERD v9 (Deep Escape) vs AdamW

This notebook applies HERD v9 (Loss-Barrier + Multi-Direction + Adaptive Step) to the NIH ChestX-ray14 dataset using DenseNet121.

**Goal**: Replace AdamW with HERD v9 and compare multi-label classification performance (AUC).

## HERD v9 Approach
1. **Warmup** — Train with AdamW for a few epochs to reach a reasonable point
2. **Lanczos Decomposition** — Compute Hessian spectrum, extract negative eigenvectors
3. **Deep Escape** — Walk along negative curvature directions using loss-barrier detection
4. **Basin Training** — Continue training in the (potentially new) basin with AdamW

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
import copy, time, json

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.amp import GradScaler, autocast

import torchvision.transforms as T
import torchvision.models as models

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from scipy import stats

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability()
    if cap[0] < 7:
        print(f"WARNING: GPU {torch.cuda.get_device_name()} (sm_{cap[0]}{cap[1]}) not supported.")
        print("Switch to T4 GPU in Kaggle: Settings -> Accelerator -> GPU T4 x2")
        DEVICE = torch.device("cpu")
    else:
        DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

print(f"Device: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
print(f"PyTorch: {torch.__version__}")

## 1. Data Loading — NIH ChestX-ray14

In [ ]:
_CANDIDATES = [
    "/kaggle/input/data",
    "/kaggle/input/datasets/organizations/nih-chest-xrays/data",
    "/kaggle/input/nih-chest-xrays-data",
]
BASE_DIR = None
for _c in _CANDIDATES:
    if os.path.isdir(_c):
        BASE_DIR = _c
        break

if BASE_DIR is None:
    for root, dirs, files in os.walk("/kaggle/input"):
        if "Data_Entry_2017.csv" in files:
            BASE_DIR = root
            break

assert BASE_DIR is not None, "Could not find dataset! Add 'nih-chest-xrays/data' as Kaggle input."
print(f"Dataset found at: {BASE_DIR}")

CSV_PATH = os.path.join(BASE_DIR, "Data_Entry_2017.csv")

IMAGE_DIRS = []
for folder in sorted(os.listdir(BASE_DIR)):
    full = os.path.join(BASE_DIR, folder)
    if not os.path.isdir(full):
        continue
    sub = os.path.join(full, "images")
    if os.path.isdir(sub):
        IMAGE_DIRS.append(sub)
    elif folder.startswith("images_"):
        IMAGE_DIRS.append(full)

print(f"Found {len(IMAGE_DIRS)} image directories")

In [ ]:
PATHOLOGIES = [
    "Atelectasis", "Cardiomegaly", "Effusion", "Infiltration",
    "Mass", "Nodule", "Pneumonia", "Pneumothorax",
    "Consolidation", "Edema", "Emphysema", "Fibrosis",
    "Pleural_Thickening", "Hernia"
]
NUM_CLASSES = len(PATHOLOGIES)

IMG_SIZE = 224
BATCH_SIZE = 128
NUM_WORKERS = 2
NUM_EPOCHS = 8
VAL_RATIO = 0.1
TEST_RATIO = 0.1
TRAIN_SAMPLE = 20000  # subsample training patients for speed (set None for full dataset)

HERD_WARMUP_EPOCHS = 3
HERD_LR = 1e-4
HERD_WD = 1e-5
HERD_LANCZOS_K = 15
HERD_MAX_ESCAPE_STEPS = 60
HERD_MIN_DISTANCE = 0.3
HERD_MAX_DIRECTIONS = 3
HERD_LOSS_PATIENCE = 4

print(f"Pathologies ({NUM_CLASSES}): {PATHOLOGIES}")
print(f"Batch size: {BATCH_SIZE}, Epochs: {NUM_EPOCHS}, Train sample: {TRAIN_SAMPLE}")
print(f"HERD v9 Config: warmup={HERD_WARMUP_EPOCHS}, lanczos_k={HERD_LANCZOS_K}, "
      f"max_steps={HERD_MAX_ESCAPE_STEPS}, min_dist={HERD_MIN_DISTANCE}")

In [ ]:
df = pd.read_csv(CSV_PATH)
print(f"Total entries: {len(df)}")

filename_to_path = {}
for img_dir in IMAGE_DIRS:
    for fname in os.listdir(img_dir):
        if fname.endswith(".png"):
            filename_to_path[fname] = os.path.join(img_dir, fname)

print(f"Indexed {len(filename_to_path)} images on disk")

df["full_path"] = df["Image Index"].map(filename_to_path)
df = df.dropna(subset=["full_path"]).reset_index(drop=True)
print(f"Matched {len(df)} entries to image files")

for pathology in PATHOLOGIES:
    df[pathology] = df["Finding Labels"].apply(lambda x: 1.0 if pathology in x.split("|") else 0.0)

patient_ids = df["Patient ID"].unique()
train_patients, temp_patients = train_test_split(
    patient_ids, test_size=VAL_RATIO + TEST_RATIO, random_state=SEED
)
val_patients, test_patients = train_test_split(
    temp_patients, test_size=TEST_RATIO / (VAL_RATIO + TEST_RATIO), random_state=SEED
)

train_df = df[df["Patient ID"].isin(train_patients)].reset_index(drop=True)
val_df   = df[df["Patient ID"].isin(val_patients)].reset_index(drop=True)
test_df  = df[df["Patient ID"].isin(test_patients)].reset_index(drop=True)

# Subsample training set for faster iteration
if TRAIN_SAMPLE is not None and len(train_df) > TRAIN_SAMPLE:
    train_df = train_df.sample(n=TRAIN_SAMPLE, random_state=SEED).reset_index(drop=True)
    print(f"Subsampled training set to {TRAIN_SAMPLE} images")

# Also limit val/test for speed
val_df = val_df.sample(n=min(5000, len(val_df)), random_state=SEED).reset_index(drop=True)
test_df = test_df.sample(n=min(5000, len(test_df)), random_state=SEED).reset_index(drop=True)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

In [ ]:
class ChestXrayDataset(Dataset):
    def __init__(self, dataframe, pathologies, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.pathologies = pathologies
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["full_path"]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        label = torch.tensor(row[self.pathologies].values.astype(np.float32))
        return img, label


train_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(10),
    T.ColorJitter(brightness=0.1, contrast=0.1),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_ds = ChestXrayDataset(train_df, PATHOLOGIES, transform=train_transform)
val_ds   = ChestXrayDataset(val_df,   PATHOLOGIES, transform=val_transform)
test_ds  = ChestXrayDataset(test_df,  PATHOLOGIES, transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")

## 2. Model — DenseNet121 (Multi-Label)

In [ ]:
class DenseNet121MultiLabel(nn.Module):
    def __init__(self, num_classes, pretrained=True):
        super().__init__()
        self.densenet = models.densenet121(
            weights=models.DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None
        )
        in_features = self.densenet.classifier.in_features
        self.densenet.classifier = nn.Sequential(
            nn.Linear(in_features, num_classes),
        )

    def forward(self, x):
        features = self.densenet.features(x)
        out = torch.relu(features)
        out = nn.functional.adaptive_avg_pool2d(out, (1, 1))
        out = torch.flatten(out, 1)
        out = self.densenet.classifier(out)
        return out


def make_model(freeze_early=True):
    model = DenseNet121MultiLabel(NUM_CLASSES, pretrained=True).to(DEVICE)
    if freeze_early:
        for name, param in model.densenet.features.named_parameters():
            if "denseblock3" in name or "denseblock4" in name or "norm5" in name:
                param.requires_grad = True
            else:
                param.requires_grad = False
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Params: {total:,} total | {trainable:,} trainable ({100*trainable/total:.1f}%)")
    return model


_ = make_model()
del _
torch.cuda.empty_cache() if DEVICE.type == 'cuda' else None

## 3. Loss & Metrics

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        probs = torch.sigmoid(logits)
        p_t = probs * targets + (1 - probs) * (1 - targets)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        focal_weight = alpha_t * (1 - p_t) ** self.gamma
        return (focal_weight * bce).mean()


def compute_auc(y_true, y_scores):
    aucs = []
    for i in range(y_true.shape[1]):
        if y_true[:, i].sum() == 0:
            continue
        try:
            auc = roc_auc_score(y_true[:, i], y_scores[:, i])
            aucs.append(auc)
        except ValueError:
            continue
    return np.mean(aucs) if aucs else 0.0


@torch.no_grad()
def evaluate_model(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    all_labels, all_preds = [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        with autocast("cuda"):
            logits = model(imgs)
            loss = criterion(logits, labels)
        running_loss += loss.item() * imgs.size(0)
        all_labels.append(labels.cpu().numpy())
        all_preds.append(torch.sigmoid(logits).cpu().numpy())
    epoch_loss = running_loss / len(loader.dataset)
    all_labels = np.concatenate(all_labels)
    all_preds = np.concatenate(all_preds)
    epoch_auc = compute_auc(all_labels, all_preds)
    return epoch_loss, epoch_auc


print("Loss & metrics ready")

## 4. HERD v9 — Hessian Tools (Lanczos + Multi-Direction Escape)

In [ ]:
def hessian_vector_product(model, loss_fn, loader, device, v, num_batches=3):
    """Compute Hv via autodiff."""
    params = [p for p in model.parameters() if p.requires_grad]
    model.zero_grad()
    total_loss = 0.0
    count = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        loss = loss_fn(model(x), y)
        total_loss += loss
        count += 1
        if count >= num_batches:
            break
    total_loss = total_loss / count

    grads = torch.autograd.grad(total_loss, params, create_graph=True)
    flat_grad = torch.cat([g.contiguous().view(-1) for g in grads])
    flat_v = torch.cat([vi.contiguous().view(-1) for vi in v])
    grad_v = torch.dot(flat_grad, flat_v)
    hvp = torch.autograd.grad(grad_v, params, retain_graph=False)
    return [h.detach() for h in hvp]


def lanczos_decomposition(model, loss_fn, loader, device, k=15, num_batches=3):
    """Full Lanczos tridiagonalization for top-k eigenvalues/eigenvectors."""
    params = [p for p in model.parameters() if p.requires_grad]

    q = [torch.randn_like(p) for p in params]
    norm = torch.sqrt(sum(torch.sum(x**2) for x in q))
    q = [x / (norm + 1e-10) for x in q]

    Q_list = [q]
    alphas = []
    betas = []
    q_prev = [torch.zeros_like(p) for p in params]
    beta_prev = 0.0

    for j in range(k):
        Hq = hessian_vector_product(model, loss_fn, loader, device, q, num_batches)
        alpha = sum(torch.sum(qi * hqi) for qi, hqi in zip(q, Hq)).item()
        alphas.append(alpha)

        r = [hqi - alpha * qi - beta_prev * qpi
             for hqi, qi, qpi in zip(Hq, q, q_prev)]

        for Q_old in Q_list:
            dot = sum(torch.sum(ri * qo) for ri, qo in zip(r, Q_old))
            r = [ri - dot.item() * qo for ri, qo in zip(r, Q_old)]

        beta = torch.sqrt(sum(torch.sum(ri**2) for ri in r)).item()

        if beta < 1e-10:
            break

        if j < k - 1:
            betas.append(beta)
            q_prev = q
            beta_prev = beta
            q = [ri / (beta + 1e-10) for ri in r]
            Q_list.append(q)

    m = len(alphas)
    T_mat = np.zeros((m, m))
    for i in range(m):
        T_mat[i, i] = alphas[i]
    for i in range(len(betas)):
        T_mat[i, i+1] = betas[i]
        T_mat[i+1, i] = betas[i]

    eigenvalues_raw, eigvecs_raw = np.linalg.eigh(T_mat)
    sort_idx = np.argsort(eigenvalues_raw)[::-1]
    eigenvalues = eigenvalues_raw[sort_idx]
    eigvec_coeffs = eigvecs_raw[:, sort_idx]

    spectrum_info = {
        'eigenvalues': eigenvalues.tolist(),
        'n_positive': int(np.sum(eigenvalues > 0.1)),
        'n_negative': int(np.sum(eigenvalues < -0.1)),
        'max_eigenvalue': float(eigenvalues[0]),
        'min_eigenvalue': float(eigenvalues[-1]),
    }

    return eigenvalues, eigvec_coeffs, Q_list, spectrum_info


def extract_negative_eigenvectors(eigenvalues, eigvec_coeffs, Q_list, threshold=-0.1):
    """Extract all eigenvectors with eigenvalue < threshold."""
    neg_eigvecs = []
    m = len(eigenvalues)
    n_basis = len(Q_list)

    for idx in range(m):
        eig_val = eigenvalues[idx]
        if eig_val >= threshold:
            continue

        coeffs = eigvec_coeffs[:, idx]
        n_use = min(len(coeffs), n_basis)

        v = [torch.zeros_like(Q_list[0][p_idx]) for p_idx in range(len(Q_list[0]))]
        for j in range(n_use):
            c = float(coeffs[j])
            for p_idx in range(len(v)):
                v[p_idx] = v[p_idx] + c * Q_list[j][p_idx]

        norm = torch.sqrt(sum(torch.sum(x**2) for x in v))
        v = [x / (norm + 1e-10) for x in v]
        neg_eigvecs.append((v, float(eig_val)))

    neg_eigvecs.sort(key=lambda x: x[1])
    return neg_eigvecs


print("Hessian tools (Lanczos + eigenvector extraction) ready")

In [ ]:
def get_param_vector(model):
    return torch.cat([p.data.view(-1) for p in model.parameters() if p.requires_grad])


def compute_train_loss_batches(model, loss_fn, loader, device, num_batches=5):
    """Deterministic loss on a few batches (eval mode)."""
    model.eval()
    total_loss = 0.0
    count = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            total_loss += loss_fn(model(x), y).item()
            count += 1
            if count >= num_batches:
                break
    return total_loss / count


def escape_along_direction(model, direction, eigenvalue, loss_fn, loader, device,
                           max_steps=60, min_distance=0.3, loss_patience=4,
                           direction_label=""):
    """
    Walk along a negative curvature direction using loss-barrier detection.
    Includes detailed step-by-step logging.
    """
    params = [p for p in model.parameters() if p.requires_grad]
    pre_params = get_param_vector(model)

    escape_lr = 1.0 / np.sqrt(abs(eigenvalue) + 1e-10)
    escape_lr = np.clip(escape_lr, 0.01, 0.5)

    start_loss = compute_train_loss_batches(model, loss_fn, loader, device, num_batches=8)

    print(f"          [{direction_label}] start_loss={start_loss:.5f}, escape_lr={escape_lr:.4f}, "
          f"eigenvalue={eigenvalue:.3f}")

    log = {
        'eigenvalue': eigenvalue,
        'escape_lr': escape_lr,
        'start_loss': start_loss,
        'barrier_crossed': False,
        'escaped': False,
        'loss_trajectory': [],
    }

    crossed_barrier = False
    below_start_count = 0
    max_loss = start_loss
    current_loss = start_loss

    for step in range(max_steps):
        with torch.no_grad():
            for p, vi in zip(params, direction):
                p.add_(vi, alpha=escape_lr)

        current_loss = compute_train_loss_batches(model, loss_fn, loader, device, num_batches=8)
        dist = torch.norm(get_param_vector(model) - pre_params).item()

        log['loss_trajectory'].append({
            'step': step + 1, 'loss': current_loss, 'dist': dist,
            'barrier': crossed_barrier, 'below_count': below_start_count
        })

        if current_loss > max_loss:
            max_loss = current_loss

        if current_loss > start_loss + 0.005:
            crossed_barrier = True
            log['barrier_crossed'] = True

        if crossed_barrier and current_loss < start_loss:
            below_start_count += 1
        else:
            below_start_count = 0

        if (crossed_barrier and below_start_count >= loss_patience and dist >= min_distance):
            log['escaped'] = True
            log['reason'] = 'loss_barrier_crossed'
            log['n_steps'] = step + 1
            log['barrier_height'] = max_loss - start_loss
            log['final_loss'] = current_loss
            print(f"          [{direction_label}] ESCAPED at step {step+1}: "
                  f"barrier_h={max_loss-start_loss:.5f}, dist={dist:.4f}, "
                  f"loss={current_loss:.5f} (was {start_loss:.5f})")
            break

        if (dist >= min_distance * 2 and current_loss < start_loss - 0.005
                and not crossed_barrier):
            log['escaped'] = True
            log['reason'] = 'monotone_descent'
            log['n_steps'] = step + 1
            log['barrier_height'] = 0.0
            log['final_loss'] = current_loss
            print(f"          [{direction_label}] ESCAPED (monotone) at step {step+1}: "
                  f"dist={dist:.4f}, loss={current_loss:.5f} (was {start_loss:.5f})")
            break

        if (step + 1) % 10 == 0:
            print(f"          [{direction_label}] step {step+1:3d}: "
                  f"loss={current_loss:.5f} (delta={current_loss-start_loss:+.5f}), "
                  f"dist={dist:.4f}, barrier={'Y' if crossed_barrier else 'N'}, "
                  f"below={below_start_count}")

    log['fall_distance'] = torch.norm(get_param_vector(model) - pre_params).item()
    if 'n_steps' not in log:
        log['n_steps'] = max_steps
        log['reason'] = 'max_steps'
        log['final_loss'] = current_loss
        print(f"          [{direction_label}] max_steps reached: "
              f"loss={current_loss:.5f}, dist={log['fall_distance']:.4f}, "
              f"barrier={'crossed' if crossed_barrier else 'not crossed'}")

    return log


def herd_v9_escape(model, loss_fn, loader, device, lanczos_k=15,
                   max_steps=60, min_distance=0.3, max_directions=3,
                   loss_patience=4):
    """
    HERD v9: Multi-direction deep escape with loss-barrier detection.
    Full logging of Lanczos spectrum and escape attempts.
    """
    saved_state = copy.deepcopy(model.state_dict())

    log = {
        'has_negative_curvature': False,
        'n_negative_directions': 0,
        'directions_tried': [],
        'escaped': False,
    }

    print(f"\n    {'='*60}")
    print(f"    HERD v9 ESCAPE — Lanczos Decomposition")
    print(f"    {'='*60}")
    print(f"    Computing Lanczos (k={lanczos_k}, num_batches=3)...")
    t_lanczos = time.time()

    eigenvalues, eigvec_coeffs, Q_list, spectrum_info = lanczos_decomposition(
        model, loss_fn, loader, device, k=lanczos_k, num_batches=3
    )

    lanczos_time = time.time() - t_lanczos
    log['spectrum'] = spectrum_info
    print(f"    Lanczos completed in {lanczos_time:.1f}s")
    print(f"    Spectrum summary:")
    print(f"      Positive eigenvalues (>0.1):  {spectrum_info['n_positive']}")
    print(f"      Negative eigenvalues (<-0.1): {spectrum_info['n_negative']}")
    print(f"      Max eigenvalue: {spectrum_info['max_eigenvalue']:.4f}")
    print(f"      Min eigenvalue: {spectrum_info['min_eigenvalue']:.4f}")
    print(f"      Top-5 eigenvalues: {[f'{e:.2f}' for e in spectrum_info['eigenvalues'][:5]]}")
    print(f"      Bottom-5 eigenvalues: {[f'{e:.2f}' for e in spectrum_info['eigenvalues'][-5:]]}")

    neg_eigvecs = extract_negative_eigenvectors(eigenvalues, eigvec_coeffs, Q_list)
    log['n_negative_directions'] = len(neg_eigvecs)

    del Q_list, eigvec_coeffs, eigenvalues
    torch.cuda.empty_cache() if device.type == 'cuda' else None

    if len(neg_eigvecs) == 0:
        log['has_negative_curvature'] = False
        log['reason'] = 'no_negative_curvature'
        log['fall_distance'] = 0.0
        print(f"\n    No negative curvature found — landscape is a bowl at this point.")
        print(f"    Skipping escape, continuing with standard training.")
        return log

    log['has_negative_curvature'] = True
    print(f"\n    {len(neg_eigvecs)} negative curvature direction(s) found:")
    for i, (_, ev) in enumerate(neg_eigvecs):
        print(f"      Direction {i+1}: lambda = {ev:.4f}")

    print(f"\n    {'='*60}")
    print(f"    ESCAPE ATTEMPTS (trying up to {max_directions} directions x 2 signs)")
    print(f"    {'='*60}")

    n_to_try = min(len(neg_eigvecs), max_directions)
    best_escape = None
    best_escape_loss = float('inf')
    best_escape_state = None

    for dir_idx in range(n_to_try):
        direction, eig_val = neg_eigvecs[dir_idx]
        print(f"\n      {'─'*50}")
        print(f"      Direction {dir_idx+1}/{n_to_try}: lambda = {eig_val:.4f}")
        print(f"      {'─'*50}")

        # Positive direction
        print(f"\n        Trying POSITIVE direction...")
        model.load_state_dict(copy.deepcopy(saved_state))
        dir_log_pos = escape_along_direction(
            model, direction, eig_val, loss_fn, loader, device,
            max_steps=max_steps, min_distance=min_distance,
            loss_patience=loss_patience, direction_label=f"D{dir_idx+1}+"
        )
        state_pos = copy.deepcopy(model.state_dict()) if dir_log_pos.get('escaped') else None

        # Negative direction
        print(f"\n        Trying NEGATIVE direction...")
        model.load_state_dict(copy.deepcopy(saved_state))
        neg_direction = [-d for d in direction]
        dir_log_neg = escape_along_direction(
            model, neg_direction, eig_val, loss_fn, loader, device,
            max_steps=max_steps, min_distance=min_distance,
            loss_patience=loss_patience, direction_label=f"D{dir_idx+1}-"
        )
        state_neg = copy.deepcopy(model.state_dict()) if dir_log_neg.get('escaped') else None

        # Pick the better side
        if dir_log_pos.get('escaped') and dir_log_neg.get('escaped'):
            if dir_log_neg.get('final_loss', 999) < dir_log_pos.get('final_loss', 999):
                chosen_log, chosen_state = dir_log_neg, state_neg
                chosen_sign = "negative"
            else:
                chosen_log, chosen_state = dir_log_pos, state_pos
                chosen_sign = "positive"
        elif dir_log_pos.get('escaped'):
            chosen_log, chosen_state = dir_log_pos, state_pos
            chosen_sign = "positive"
        elif dir_log_neg.get('escaped'):
            chosen_log, chosen_state = dir_log_neg, state_neg
            chosen_sign = "negative"
        else:
            chosen_log, chosen_state = dir_log_pos, None
            chosen_sign = "none (both failed)"

        log['directions_tried'].append({
            'eigenvalue': eig_val,
            'escaped': chosen_log.get('escaped', False),
            'reason': chosen_log.get('reason', '?'),
            'fall_distance': chosen_log.get('fall_distance', 0),
            'barrier_height': chosen_log.get('barrier_height', 0),
            'final_loss': chosen_log.get('final_loss', None),
            'n_steps': chosen_log.get('n_steps', 0),
            'chosen_sign': chosen_sign,
        })

        escaped_str = 'ESCAPED' if chosen_log.get('escaped') else 'NO ESCAPE'
        print(f"\n        >> Direction {dir_idx+1} result: {escaped_str}")
        print(f"           Chosen sign: {chosen_sign}")
        print(f"           Reason: {chosen_log.get('reason', '?')}")
        print(f"           Distance: {chosen_log.get('fall_distance', 0):.4f}")
        print(f"           Steps: {chosen_log.get('n_steps', 0)}")

        if chosen_log.get('escaped') and chosen_state is not None:
            final_loss = chosen_log.get('final_loss', float('inf'))
            if final_loss < best_escape_loss:
                best_escape = chosen_log
                best_escape_loss = final_loss
                best_escape_state = chosen_state

    # Final decision
    print(f"\n    {'='*60}")
    print(f"    ESCAPE VERDICT")
    print(f"    {'='*60}")

    if best_escape is not None:
        model.load_state_dict(best_escape_state)
        log['escaped'] = True
        log['fall_distance'] = best_escape.get('fall_distance', 0)
        log['reason'] = best_escape.get('reason', '?')
        print(f"    ESCAPED SUCCESSFULLY")
        print(f"      Method: {best_escape.get('reason')}")
        print(f"      Distance moved: {best_escape.get('fall_distance', 0):.4f}")
        print(f"      Barrier height: {best_escape.get('barrier_height', 0):.5f}")
        print(f"      Loss improvement: {best_escape.get('start_loss', 0) - best_escape.get('final_loss', 0):.5f}")
    else:
        model.load_state_dict(saved_state)
        log['escaped'] = False
        log['fall_distance'] = 0.0
        log['reason'] = 'no_direction_escaped'
        print(f"    NO ESCAPE — all directions failed")
        print(f"    Model restored to pre-escape state")

    print(f"    {'='*60}\n")
    return log


print("HERD v9 escape engine ready (with thorough logging)")

## 5. Training Functions

In [ ]:
def train_epochs(model, loader, val_loader, criterion, n_epochs, lr=1e-4, wd=1e-5, tag=""):
    """Standard training with AdamW + cosine schedule + mixed precision + detailed logging."""
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                           lr=lr, weight_decay=wd)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
    scaler = GradScaler()

    best_auc = 0.0
    best_state = None
    history = {'train_loss': [], 'val_loss': [], 'val_auc': [], 'lr': []}

    print(f"\n    {'Epoch':<8} {'TrainLoss':>10} {'ValLoss':>10} {'ValAUC':>10} {'BestAUC':>10} {'LR':>12} {'Time':>8}")
    print(f"    {'-'*70}")

    for epoch in range(1, n_epochs + 1):
        ep_start = time.time()
        model.train()
        running_loss = 0.0
        n_batches = 0

        for imgs, labels in tqdm(loader, desc=f"{tag} ep{epoch}", leave=False):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            with autocast("cuda"):
                logits = model(imgs)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item() * imgs.size(0)
            n_batches += 1

        current_lr = scheduler.get_last_lr()[0]
        scheduler.step()

        train_loss = running_loss / len(loader.dataset)
        val_loss, val_auc = evaluate_model(model, val_loader, criterion)

        if val_auc > best_auc:
            best_auc = val_auc
            best_state = copy.deepcopy(model.state_dict())

        ep_time = time.time() - ep_start
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_auc'].append(val_auc)
        history['lr'].append(current_lr)

        marker = " *" if val_auc == best_auc else ""
        print(f"    {epoch:<8} {train_loss:>10.4f} {val_loss:>10.4f} {val_auc:>10.4f} "
              f"{best_auc:>10.4f} {current_lr:>12.6f} {ep_time:>7.1f}s{marker}")

    print(f"    {'-'*70}")
    print(f"    Best Val AUC: {best_auc:.4f}")
    return {'best_auc': best_auc, 'best_state': best_state, 'history': history}


print("Training functions ready (with detailed per-epoch logging)")

## 6. Run HERD v9 (Deep Escape)

In [ ]:
print("=" * 70)
print("HERD v9: Deep Escape -- DenseNet121 on NIH ChestX-ray14")
print("=" * 70)

criterion = FocalLoss(alpha=0.25, gamma=2.0)

torch.manual_seed(SEED)
np.random.seed(SEED)
model_herd = make_model(freeze_early=True)

t0 = time.time()

# Phase 1: Warmup
print(f"\n--- Phase 1: Warmup ({HERD_WARMUP_EPOCHS} epochs) ---")
warmup_result = train_epochs(
    model_herd, train_loader, val_loader, criterion,
    n_epochs=HERD_WARMUP_EPOCHS, lr=HERD_LR, wd=HERD_WD, tag="HERD-warmup"
)
warmup_auc = warmup_result['best_auc']
print(f"    Post-warmup Val AUC: {warmup_auc:.4f}")

# Phase 2: HERD v9 Deep Escape
print(f"\n--- Phase 2: HERD v9 Deep Escape ---")
escape_log = herd_v9_escape(
    model_herd, criterion, train_loader, DEVICE,
    lanczos_k=HERD_LANCZOS_K,
    max_steps=HERD_MAX_ESCAPE_STEPS,
    min_distance=HERD_MIN_DISTANCE,
    max_directions=HERD_MAX_DIRECTIONS,
    loss_patience=HERD_LOSS_PATIENCE,
)

print(f"\n    Escape result: neg_curvature={'Yes' if escape_log['has_negative_curvature'] else 'No'}, "
      f"escaped={'Yes' if escape_log['escaped'] else 'No'}, "
      f"distance={escape_log.get('fall_distance', 0):.4f}")

# Phase 3: Basin Training (remaining epochs)
remaining_epochs = NUM_EPOCHS - HERD_WARMUP_EPOCHS
print(f"\n--- Phase 3: Basin Training ({remaining_epochs} epochs) ---")
basin_result = train_epochs(
    model_herd, train_loader, val_loader, criterion,
    n_epochs=remaining_epochs, lr=HERD_LR, wd=HERD_WD, tag="HERD-basin"
)

herd_time = time.time() - t0

herd_best_auc = max(warmup_auc, basin_result['best_auc'])
best_state = basin_result['best_state'] if basin_result['best_auc'] >= warmup_auc else warmup_result['best_state']
model_herd.load_state_dict(best_state)

herd_test_loss, herd_test_auc = evaluate_model(model_herd, test_loader, criterion)

print(f"\nHERD v9 Results:")
print(f"  Best Val AUC:  {herd_best_auc:.4f}")
print(f"  Test AUC:      {herd_test_auc:.4f}")
print(f"  Time:          {herd_time:.0f}s")
print(f"  Escaped:       {escape_log['escaped']}")

## 7. Results & Per-Class AUC

In [ ]:
def get_per_class_auc(model, loader):
    model.eval()
    all_labels, all_preds = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(DEVICE)
            with autocast("cuda"):
                logits = model(imgs)
            all_labels.append(labels.numpy())
            all_preds.append(torch.sigmoid(logits).cpu().numpy())
    all_labels = np.concatenate(all_labels)
    all_preds = np.concatenate(all_preds)

    per_class = {}
    for i, pathology in enumerate(PATHOLOGIES):
        if all_labels[:, i].sum() == 0:
            continue
        try:
            per_class[pathology] = roc_auc_score(all_labels[:, i], all_preds[:, i])
        except ValueError:
            pass
    return per_class


print("=" * 70)
print("HERD v9 — FINAL RESULTS")
print("=" * 70)

herd_per_class = get_per_class_auc(model_herd, test_loader)

print(f"\n{'Pathology':<25} {'Test AUC':>10}")
print("-" * 38)
for p in PATHOLOGIES:
    auc = herd_per_class.get(p, 0)
    print(f"{p:<25} {auc:>10.4f}")
print("-" * 38)
mean_auc = np.mean(list(herd_per_class.values()))
print(f"{'MEAN':<25} {mean_auc:>10.4f}")

print(f"\nSummary:")
print(f"  Best Val AUC:  {herd_best_auc:.4f}")
print(f"  Test AUC:      {herd_test_auc:.4f}")
print(f"  Escaped:       {escape_log['escaped']}")
print(f"  Total time:    {herd_time:.0f}s")

## 8. Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Per-class AUC bar chart
ax = axes[0]
x = np.arange(NUM_CLASSES)
vals = [herd_per_class.get(p, 0) for p in PATHOLOGIES]
bars = ax.bar(x, vals, color='#e74c3c', alpha=0.7, edgecolor='black', linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels([p[:10] for p in PATHOLOGIES], rotation=45, ha='right', fontsize=8)
ax.set_ylabel('AUC')
ax.set_title('HERD v9 — Per-Class Test AUC (NIH ChestX-ray14)')
ax.set_ylim(0.5, 1.0)
ax.axhline(y=mean_auc, color='black', linestyle='--', alpha=0.5, label=f'Mean={mean_auc:.4f}')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Escape trajectory info
ax2 = axes[1]
ax2.axis('off')
info_text = (
    f"HERD v9 Deep Escape — NIH ChestX-ray14\n"
    f"{'='*45}\n\n"
    f"Model: DenseNet121 (partial freeze)\n"
    f"Train samples: {len(train_df)}\n"
    f"Epochs: {HERD_WARMUP_EPOCHS} warmup + {NUM_EPOCHS - HERD_WARMUP_EPOCHS} basin\n\n"
    f"Escape Details:\n"
    f"  Negative curvature: {'Yes' if escape_log['has_negative_curvature'] else 'No'}\n"
    f"  Directions tried: {len(escape_log.get('directions_tried', []))}\n"
    f"  Escaped: {'Yes' if escape_log['escaped'] else 'No'}\n"
    f"  Reason: {escape_log.get('reason', 'N/A')}\n"
    f"  Fall distance: {escape_log.get('fall_distance', 0):.4f}\n\n"
    f"Results:\n"
    f"  Val AUC (best): {herd_best_auc:.4f}\n"
    f"  Test AUC:       {herd_test_auc:.4f}\n"
    f"  Total time:     {herd_time:.0f}s"
)
ax2.text(0.05, 0.95, info_text, transform=ax2.transAxes, fontsize=10,
         verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.savefig('herd_v9_nih_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: herd_v9_nih_results.png")

## 9. Save Model & Results

In [ ]:
results = {
    'experiment': 'HERD v9 on NIH ChestX-ray14',
    'model': 'DenseNet121 (pretrained, partial freeze)',
    'dataset': 'NIH ChestX-ray14 (14 pathologies, multi-label)',
    'config': {
        'total_epochs': NUM_EPOCHS,
        'warmup_epochs': HERD_WARMUP_EPOCHS,
        'lr': HERD_LR,
        'weight_decay': HERD_WD,
        'batch_size': BATCH_SIZE,
        'train_samples': len(train_df),
        'lanczos_k': HERD_LANCZOS_K,
        'max_escape_steps': HERD_MAX_ESCAPE_STEPS,
        'min_distance': HERD_MIN_DISTANCE,
        'max_directions': HERD_MAX_DIRECTIONS,
    },
    'results': {
        'best_val_auc': float(herd_best_auc),
        'test_auc': float(herd_test_auc),
        'per_class_auc': {k: float(v) for k, v in herd_per_class.items()},
        'time_seconds': herd_time,
    },
    'escape_log': {
        'has_negative_curvature': escape_log['has_negative_curvature'],
        'escaped': escape_log['escaped'],
        'reason': escape_log.get('reason', 'N/A'),
        'fall_distance': escape_log.get('fall_distance', 0),
        'n_negative_directions': escape_log.get('n_negative_directions', 0),
        'n_directions_tried': len(escape_log.get('directions_tried', [])),
        'directions_tried': escape_log.get('directions_tried', []),
        'spectrum': escape_log.get('spectrum', {}),
    },
}

with open('herd_v9_nih_results.json', 'w') as f:
    json.dump(results, f, indent=2)

torch.save({
    'model_state_dict': best_state,
    'pathologies': PATHOLOGIES,
    'test_auc': herd_test_auc,
    'per_class_auc': herd_per_class,
    'escape_log': escape_log,
}, 'best_densenet121_nih_herd_v9.pth')

print("Saved: herd_v9_nih_results.json")
print("Saved: best_densenet121_nih_herd_v9.pth")
print(f"\nFinal Test AUC: {herd_test_auc:.4f} (Mean per-class: {mean_auc:.4f})")